# Traffic sign workflow — Kaggle GPU → offline reports

Run All trains **one YOLOv8s** model, uploads best weights/context to W&B, then evaluates and renders detections in fresh processes. Finally it exports a named report ZIP. Select a GPU accelerator and enable Internet. Grant access to the `WANDB_API_KEY` Kaggle Secret.

Optional commands are fully commented: smoke training, tuning, additional models, resume and public-test predictions. Uncomment a whole block. Change dataset mount and repository ref below. Push the implementation before cloning it on Kaggle.

Training validation remains enabled for best-checkpoint selection. Small-object evaluation runs only in `evaluate.py`. Training plots and weights are excluded from the report ZIP.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from uuid import uuid4

DATA_ROOT = Path('/kaggle/input/datasets/ducmonday/zalo-traffic-sign/za_traffic_2020')
REPO_DIR = Path('/kaggle/working/AIT3001.Group6.traffic.sign.detection')
REPO_REF = 'aduc'  # Branch or commit containing this workflow.
PREPARED_DIR = Path('/kaggle/working/prepared_traffic_signs')
OUTPUT_ROOT = Path('/kaggle/working/traffic_sign_runs')
EXPORT_DIR = Path('/kaggle/working/exports')
MODEL_RECIPE = 'yolov8'  # yolov8, yolov8m, yolov8l
RUN_LABEL = {'yolov8': 'baseline-yolov8s', 'yolov8m': 'baseline-yolov8m', 'yolov8l': 'baseline-yolov8l'}[MODEL_RECIPE]
TRAIN_BATCH = {'yolov8': 16, 'yolov8m': 4, 'yolov8l': 2}[MODEL_RECIPE]  # Reduce further if needed.
TRACKING_MODE = 'online'  # W&B evaluation requires uploaded model + context.
SESSION_DIR = Path('/kaggle/working/workflow_logs') / (datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-' + uuid4().hex[:6])
SESSION_DIR.mkdir(parents=True)
for variable, name in (('WANDB_DATA_DIR', 'wandb-data'), ('WANDB_CACHE_DIR', 'wandb-cache'),
                       ('WANDB_CONFIG_DIR', 'wandb-config'), ('MPLCONFIGDIR', 'matplotlib'),
                       ('YOLO_CONFIG_DIR', 'ultralytics')):
    directory = Path('/kaggle/working/.workflow-cache') / name
    directory.mkdir(parents=True, exist_ok=True)
    os.environ[variable] = str(directory)
assert (DATA_ROOT / 'traffic_train' / 'annotation.csv').is_file(), DATA_ROOT
assert (DATA_ROOT / 'traffic_train' / 'images').is_dir(), DATA_ROOT


## Install and preflight
An existing checkout is reused without resetting local changes. Check out a commit for reproducibility. Restart the kernel after installation if you imported ML libraries earlier, then rerun the configuration cell.


In [ ]:
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '-b aduc'
        'https://github.com/dohoangminh-vn/AIT3001.Group6.traffic.sign.detection.git',
        str(REPO_DIR),
    ], check=True)
subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
if TRACKING_MODE == 'online':
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
assert TRACKING_MODE in {'online', 'offline'}
# GPU checks run in a subprocess; the notebook kernel never holds a model.
subprocess.run([
    sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"; print(torch.cuda.get_device_name(0))',
], check=True)


## Command runner
Each command prints its complete overrides and saves stdout/stderr. Training and each analysis stage run in separate processes. A failed process raises immediately. Logs are copied into the selected run for export.


In [ ]:
RUN_DIR = None

def run_command(arguments, stage):
    command = [str(item) for item in arguments]
    print(shlex.join(command), flush=True)
    log = SESSION_DIR / f'{stage}-{uuid4().hex[:6]}.log'
    environment = dict(os.environ, PYTHONUNBUFFERED='1')
    with log.open('w', encoding='utf-8') as stream:
        stream.write(shlex.join(command) + '\n')
        with subprocess.Popen(command, cwd=REPO_DIR, env=environment, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace') as process:
            for line in process.stdout:
                print(line, end='')
                stream.write(line)
            returncode = process.wait()
    if RUN_DIR is not None and stage not in {'smoke', 'prepare', 'train-fasterrcnn'}:
        (RUN_DIR / 'logs').mkdir(parents=True, exist_ok=True)
        shutil.copyfile(log, RUN_DIR / 'logs' / log.name)
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

def train_run(arguments):
    global RUN_DIR
    RUN_DIR = None
    before = set(OUTPUT_ROOT.glob('*/run_manifest.json'))
    before_logs = set(SESSION_DIR.glob('*.log'))
    try:
        run_command(arguments, 'train')
    except subprocess.CalledProcessError as error:
        created = set(OUTPUT_ROOT.glob('*/run_manifest.json')) - before
        for path in created:
            (path.parent / 'logs').mkdir(exist_ok=True)
            for log in set(SESSION_DIR.glob('*.log')) - before_logs:
                shutil.copyfile(log, path.parent / 'logs' / log.name)
        raise
    created = set(OUTPUT_ROOT.glob('*/run_manifest.json')) - before
    assert len(created) == 1, f'Expected one new run, found {created}; select RUN_DIR explicitly.'
    RUN_DIR = next(iter(created)).parent
    (RUN_DIR / 'logs').mkdir(exist_ok=True)
    for log in set(SESSION_DIR.glob('*.log')) - before_logs:
        shutil.copyfile(log, RUN_DIR / 'logs' / log.name)
    return RUN_DIR


## Prepare the grouped street split
Source images and CSV are read-only. If CSV, seed or validation fraction changes, use a new `PREPARED_DIR`.


In [ ]:
run_command([
    sys.executable, 'train.py',
    'prepare_only=true',
    f'data.root={DATA_ROOT}',
    f'data.prepared_dir={PREPARED_DIR}',
    'seed=42',
    'data.val_fraction=0.2',
], 'prepare')


## Optional smoke training
Separate smoke output prevents mixing it into the full-run report. Validation still uses the full validation split. Smoke training does not automatically run evaluation.


In [ ]:
run_command([
    sys.executable, 'train.py',
    f'model={MODEL_RECIPE}',
    f'data.root={DATA_ROOT}',
    f'data.prepared_dir={PREPARED_DIR}',
    f'output_dir={OUTPUT_ROOT / "smoke"}',
    'tracking.name=smoke-yolov8',
    f'tracking.mode={TRACKING_MODE}',
    'tracking.checkpoint_every=1',
    'augmentation.enabled=false',
    'model.epochs=2',
    'model.fraction=0.02',
    'model.batch_size=2',
    'model.device=0',
    'evaluation.inference.score_threshold=0.001',
], 'smoke')


## Train one selected model
Starting batch sizes are s=16, m=4, l=2; these are not VRAM guarantees. Reduce TRAIN_BATCH further if needed. Each call creates a unique run name. Normal training exits before the evaluation process starts.


In [ ]:
# RUN_DIR = train_run([
#     sys.executable, 'train.py',
#     f'model={MODEL_RECIPE}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'output_dir={OUTPUT_ROOT}',
#     f'tracking.name={RUN_LABEL}',
#     f'tracking.mode={TRACKING_MODE}',
#     'tracking.checkpoint_every=5',
#     'seed=42',
#     'data.val_fraction=0.2',
#     'data.image_size=640',
#     'augmentation.enabled=false',
#     'model.epochs=50',
#     'model.patience=10',
#     f'model.batch_size={TRAIN_BATCH}',
#     'model.device=0',
#     'model.workers=4',
#     'model.amp=true',
#     'evaluation.inference.score_threshold=0.001',
#     'evaluation.inference.nms_iou=0.7',
#     'evaluation.inference.max_detections=300',
# ])


## Select the exact source run
For a run already present in working, uncomment `RUN_DIR` below. Never infer the source by latest modification time. Online upload must finish before evaluation. For offline training, sync the W&B directory first and provide the uploaded model artifact URI in the old-run cell.


In [ ]:
# # RUN_DIR = OUTPUT_ROOT / 'paste-exact-run-directory-name'
# manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
# RUN_NAME = manifest['run_name']
# ARTIFACT_URI = manifest['artifact_uri']
# print('Run name:', RUN_NAME)
# print('Run directory:', RUN_DIR)
# print('W&B:', manifest['wandb_url'])
# print('Best model:', ARTIFACT_URI)
# assert ARTIFACT_URI, 'Upload/sync best model and context to W&B before running evaluation.'


## Evaluate best model from W&B
The artifact contains weights, original config and split metadata. Evaluation checks dataset/split identity. Batch 1 is independent of training batch. AP uses confidence 0.001; operating P/R/F1 use 0.25 and matching IoU 0.5. Small-object AP uses original-image pixels.


In [ ]:
# run_command([
#     sys.executable, 'evaluate.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'tracking.mode={TRACKING_MODE}',
#     'evaluation.batch_size=1',
#     'evaluation.workers=0',
#     'evaluation.device=0',
#     'evaluation.amp=true',
#     'evaluation.conf=0.25',
#     'evaluation.match_iou=0.5',
# ], 'evaluate-baseline')


## Baseline diagnostics
Exports every error image and up to 30 deterministic samples without errors. A class mistake is one diagnostic event; operating metrics count it as one FP and one FN.


In [ ]:
# run_command([
#     sys.executable, 'diagnose.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     'diagnostic.batch_size=1',
#     'diagnostic.device=0',
#     'diagnostic.amp=true',
#     'diagnostic.conf=0.25',
#     'diagnostic.match_iou=0.5',
#     'diagnostic.correct_samples=30',
# ], 'diagnostics-baseline')


## Optional threshold tuning
Uncomment this block and the next block together. Selects NMS IoU by validation mAP50:95, then confidence by micro-F1. Confidence filtering reuses CPU predictions. The resulting scores are validation-selected, not independent test performance.


In [ ]:
# run_command([
#     sys.executable, 'tune.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     'tuning.thresholds=[nms_iou,conf]',
#     'tuning.nms_iou.values=[0.50,0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90,0.95]',
#     'tuning.nms_iou.metric=map50_95',
#     'tuning.conf.values=[0.1,0.2,0.3,0.4,0.5]',
#     'tuning.conf.metric=f1',
#     'tuning.match_iou=0.5',
#     'tuning.batch_size=1',
#     'tuning.device=0',
#     'tuning.amp=true',
# ], 'tune')
# manifest = json.loads((RUN_DIR / 'run_manifest.json').read_text())
# tuning_stage = manifest['stages']['tuning']
# assert tuning_stage['status'] == 'complete', tuning_stage
# THRESHOLDS_FILE = RUN_DIR / tuning_stage['path'] / 'best_thresholds.json'
# print('Selected thresholds:', THRESHOLDS_FILE)


In [ ]:
# run_command([
#     sys.executable, 'evaluate.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'analysis.thresholds_file={THRESHOLDS_FILE}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'tracking.mode={TRACKING_MODE}',
#     'evaluation.batch_size=1',
#     'evaluation.workers=0',
#     'evaluation.device=0',
#     'evaluation.amp=true',
# ], 'evaluate-tuned')
# run_command([
#     sys.executable, 'diagnose.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'analysis.thresholds_file={THRESHOLDS_FILE}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     'diagnostic.batch_size=1',
#     'diagnostic.device=0',
#     'diagnostic.amp=true',
#     'diagnostic.correct_samples=30',
# ], 'diagnostics-tuned')


## Optional: evaluate a W&B artifact in a new Kaggle session
Run setup/preflight and command-runner cells, skip training. Paste the exact **versioned** model URI and source run name. Run this cell, then baseline diagnostics and export. New workflow artifacts include config and split context; older weights-only artifacts need republishing with context.


In [ ]:
# RUN_NAME = 'paste-exact-source-run-name'
# RUN_DIR = OUTPUT_ROOT / RUN_NAME
# ARTIFACT_URI = 'wandb://entity/project/run_RUNID_model:v1'
# run_command([
#     sys.executable, 'evaluate.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'tracking.mode={TRACKING_MODE}',
#     'evaluation.batch_size=1',
#     'evaluation.device=0',
#     'evaluation.workers=0',
#     'evaluation.amp=true',
# ], 'evaluate-existing')


## Optional additional model or resume
Each uncommented training command creates a new run. Afterwards rerun **Select the exact source run → Evaluate → Diagnostics → Export** for that run. No automatic multi-model sweep.


In [ ]:
# RUN_DIR = train_run([
#     sys.executable, 'train.py',
#     'model=yolov8m',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'output_dir={OUTPUT_ROOT}',
#     'tracking.name=baseline-yolov8m',
#     f'tracking.mode={TRACKING_MODE}',
#     'seed=42',
#     'data.image_size=640',
#     'augmentation.enabled=false',
#     'model.epochs=50',
#     'model.batch_size=4',
#     'model.device=0',
#     'evaluation.inference.score_threshold=0.001',
#     'evaluation.inference.nms_iou=0.7',
# ])


In [ ]:
# RUN_DIR = train_run([
#     sys.executable, 'train.py',
#     'model=yolov8l',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'output_dir={OUTPUT_ROOT}',
#     'tracking.name=baseline-yolov8l',
#     f'tracking.mode={TRACKING_MODE}',
#     'seed=42',
#     'data.image_size=640',
#     'augmentation.enabled=false',
#     'model.epochs=50',
#     'model.batch_size=2',
#     'model.device=0',
#     'evaluation.inference.score_threshold=0.001',
#     'evaluation.inference.nms_iou=0.7',
# ])


In [ ]:
# Resume needs a periodic optimizer checkpoint, not the best-model artifact.
# RUN_DIR = train_run([
#     sys.executable, 'train.py',
#     'resume=entity/project/run_RUNID_checkpoint:latest',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'output_dir={OUTPUT_ROOT}',
#     'tracking.mode=online',
#     'model.batch_size=2',
#     'model.device=0',
# ])


## Optional public-test predictions
No labels are available: this produces visual predictions only, no mAP. It resolves the same pinned W&B source and saves images under the run. Run after baseline evaluation. Export again afterwards.


In [ ]:
# run_command([
#     sys.executable, 'predict.py',
#     f'analysis.checkpoint={ARTIFACT_URI}',
#     f'analysis.run_dir={RUN_DIR}',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     'diagnostic.batch_size=1',
#     'diagnostic.device=0',
#     'diagnostic.conf=0.25',
# ], 'predict-public-test')


## Optional Faster R-CNN training only
The W&B-based evaluate/tune/diagnostics/report workflow above supports YOLO only. Run Faster R-CNN separately; it retains its existing native evaluation behavior.


In [ ]:
# run_command([
#     sys.executable, 'train.py',
#     'model=fasterrcnn',
#     f'data.root={DATA_ROOT}',
#     f'data.prepared_dir={PREPARED_DIR}',
#     f'output_dir={OUTPUT_ROOT / "fasterrcnn"}',
#     'tracking.name=baseline-fasterrcnn',
#     f'tracking.mode={TRACKING_MODE}',
#     'augmentation.enabled=false',
#     'data.image_size=640',
#     'model.epochs=50',
#     'model.batch_size=2',
#     'model.device=0',
# ], 'train-fasterrcnn')


## Export this run — CPU only
Creates `index.html` with best-model metrics, small-object plots, optional tuning and detection gallery. **No training charts, README or weights.** Missing/failed stages remain explicit in the manifest and HTML. This cell can also export partial results after a failed stage. Download the named ZIP manually from Kaggle notebook outputs, or your usual Kaggle kernel download flow; unzip and open `index.html` locally.


In [ ]:
assert RUN_DIR is not None and (RUN_DIR / 'run_manifest.json').is_file()
before = set(EXPORT_DIR.glob('*.zip'))
run_command([
    sys.executable, 'export_reports.py',
    f'analysis.run_dir={RUN_DIR}',
    f'analysis.export_dir={EXPORT_DIR}',
], 'export')
created = set(EXPORT_DIR.glob('*.zip')) - before
assert len(created) == 1, created
REPORT_ZIP = next(iter(created))
print('Run:', json.loads((RUN_DIR / 'run_manifest.json').read_text())['run_name'])
print('Download:', REPORT_ZIP)
print('Size:', round(REPORT_ZIP.stat().st_size / 1024 ** 2, 2), 'MiB')
